In [ ]:
import networkx as nx
import numpy as np
import random
from collections import defaultdict

def build_uvr_graph(num_nodes=1000, k_neighbors=10, randomness=0.1):
    """
    Creates a Watts-Strogatz small-world graph. 
    This is a good proxy for a decentralized ring with local connections 
    and a few random 'shortcuts' to distant nodes.
    """
    return nx.watts_strogatz_graph(num_nodes, k_neighbors, randomness)

def precompute_distances(G, target_node):
    """
    Simulates your 'relative distance' metric. 
    In a real system, this is an estimate. Here, we use actual shortest path lengths.
    """
    return dict(nx.single_target_shortest_path_length(G, target_node))

def single_walker_step(G, current_node, distances, beta):
    """
    Calculates the probabilities and selects the next hop using gradient biasing.
    """
    neighbors = list(G.neighbors(current_node))
    
    # Calculate unnormalized weights using the biasing function: e^(-beta * distance)
    weights = [np.exp(-beta * distances.get(neighbor, float('inf'))) for neighbor in neighbors]
    
    # Normalize weights to create a probability distribution
    total_weight = sum(weights)
    if total_weight == 0:
        return random.choice(neighbors) # Fallback to pure random if weights fail
    
    probabilities = [w / total_weight for w in weights]
    
    # Choose next node based on the biased probabilities
    next_node = np.random.choice(neighbors, p=probabilities)
    return next_node

def simulate_k_walkers(G, start_node, target_node, distances, beta, k, h):
    """
    Simulates k independent random walkers attempting to find the target.
    Returns a tuple: (Success Boolean, Total Hops/Overhead)
    """
    total_overhead = 0
    
    for _ in range(k):
        current_node = start_node
        
        for step in range(h):
            total_overhead += 1
            if current_node == target_node:
                return True, total_overhead
            
            current_node = single_walker_step(G, current_node, distances, beta)
            
    return False, total_overhead

def run_parameter_sweep(G, iterations=100):
    """
    Tests different combinations of beta, k, and h to find the optimal setup.
    """
    nodes = list(G.nodes())
    
    # Define the parameter space to explore
    betas = [0.0, 0.5, 1.0, 2.0]  # 0.0 is pure random, 2.0 is highly greedy
    walker_counts = [1, 5, 15, 30] # k
    ttls = [10, 20, 50]           # h
    
    results = defaultdict(list)

    print(f"{'Beta':<6} | {'k':<4} | {'TTL (h)':<8} | {'Success Rate':<15} | {'Avg Overhead'}")
    print("-" * 60)

    for beta in betas:
        for k in walker_counts:
            for h in ttls:
                successes = 0
                total_overhead_all_runs = 0
                
                for _ in range(iterations):
                    target_node = random.choice(nodes)
                    start_node = random.choice(nodes)
                    while start_node == target_node:
                        start_node = random.choice(nodes)
                        
                    distances = precompute_distances(G, target_node)
                    
                    success, overhead = simulate_k_walkers(G, start_node, target_node, distances, beta, k, h)
                    
                    if success:
                        successes += 1
                    total_overhead_all_runs += overhead
                
                success_rate = successes / iterations
                avg_overhead = total_overhead_all_runs / iterations
                
                print(f"{beta:<6} | {k:<4} | {h:<8} | {success_rate:<15.0%} | {avg_overhead:.1f}")

print("Generating UVR Ring topology...")
# Simulating a 1,000 node network where each node connects to 10 peers
uvr_graph = build_uvr_graph(num_nodes=1000, k_neighbors=10, randomness=0.1)

print("Running parameter sweep (this may take a moment)...\n")
run_parameter_sweep(uvr_graph, iterations=100)

Generating UVR Ring topology...
Running parameter sweep (this may take a moment)...

Beta   | k    | TTL (h)  | Success Rate    | Avg Overhead
------------------------------------------------------------


AttributeError: 'generator' object has no attribute 'get'